### Notebook 14 — model_deployment

##### Purpose

Deploy the registered Candidate Transformer model to a Databricks Model Serving endpoint.

This notebook should not retrain, re-register, or repackage anything.

Its responsibility is only:

``` text

Find Candidate model version
        ↓
Check whether endpoint already exists
        ↓
Create or update serving endpoint
        ↓
Wait for deployment
        ↓
Verify endpoint state

```

Databricks custom model serving endpoints can serve a Unity Catalog model version as a served_entity. Current Databricks APIs use served_entities; the older served_models terminology is deprecated.

##### 1. Imports

In [0]:
import time

import mlflow

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

from mlflow import MlflowClient

from databricks.sdk.errors import ResourceDoesNotExist

from src.project_config import (
    REGISTERED_MODEL_NAME,
    SERVING_ENDPOINT_NAME,
    SERVING_ENTITY_NAME,
    SERVING_WORKLOAD_SIZE,
    SERVING_SCALE_TO_ZERO,
)

##### 2. Set Unity Catalog registry

In [0]:
mlflow.set_registry_uri(
    "databricks-uc"
)

registry_client = MlflowClient(
    registry_uri="databricks-uc"
)

workspace_client = WorkspaceClient()

##### 3. Resolve the Candidate dynamically

In [0]:
candidate_model = (
    registry_client
    .get_model_version_by_alias(
        name=REGISTERED_MODEL_NAME,
        alias="Candidate",
    )
)

candidate_version = str(
    candidate_model.version
)

print(
    "Registered model:",
    REGISTERED_MODEL_NAME
)

print(
    "Candidate version:",
    candidate_version
)

##### 4. Define endpoint configuration

In [0]:
endpoint_config = (
    EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                name=SERVING_ENTITY_NAME,
                entity_name= REGISTERED_MODEL_NAME,
                entity_version= candidate_version,
                workload_size= SERVING_WORKLOAD_SIZE,
                scale_to_zero_enabled= SERVING_SCALE_TO_ZERO,
            )
        ]
    )
)

name
→ name of this deployment inside the endpoint

entity_name
→ registered Unity Catalog model

entity_version
→ specific model version to serve

workload_size
→ serving compute size

scale_to_zero_enabled
→ whether compute can shut down while idle


``` text

Serving endpoint
support-ticket-transformer-endpoint
        ↓
served entity
support-ticket-transformer
        ↓
Unity Catalog model
dbw_agentic_ai_dev.support_ticket_ai
.support_ticket_transformer_classifier
        ↓
Candidate version 2

```

##### 5. Check whether endpoint already exists

In [0]:

endpoint_exists = False

try:

    existing_endpoint = (
        workspace_client
        .serving_endpoints
        .get(
            SERVING_ENDPOINT_NAME
        )
    )

    endpoint_exists = True

    print(
        "Existing endpoint found:",
        SERVING_ENDPOINT_NAME
    )

except ResourceDoesNotExist:

    print(
        "Endpoint does not exist yet."
    )

Try to get endpoint
        ↓
Exists
→ endpoint_exists = True

Does not exist
→ catch ResourceDoesNotExist
→ endpoint_exists remains False
→ continue to create endpoint

##### 7. Create OR update

In [0]:
if not endpoint_exists:

    print(
        "Creating serving endpoint..."
    )

    workspace_client.serving_endpoints.create(
        name=SERVING_ENDPOINT_NAME,
        config=endpoint_config,
    )

else:

    print(
        "Updating existing endpoint..."
    )

    workspace_client.serving_endpoints.update_config(
        name=SERVING_ENDPOINT_NAME,
        served_entities=(
            endpoint_config.served_entities
        ),
    )

##### 8. Wait for endpoint deployment


``` text
create/update request
       ↓
Databricks starts deployment
       ↓
container/environment creation
       ↓
model artifact loading
       ↓
serving process starts
       ↓
READY

```

The Transformer package includes PyTorch, Transformers, tokenizer files and fine-tuned model weights, so provisioning can take a few minutes.

In [0]:
import time

print(
    "Waiting for endpoint deployment..."
)

max_wait_seconds = 1200
poll_interval_seconds = 20
elapsed = 0

while elapsed < max_wait_seconds:

    # Refresh endpoint state
    endpoint = (
        workspace_client
        .serving_endpoints
        .get(
            SERVING_ENDPOINT_NAME
        )
    )

    # Current endpoint readiness
    ready_state = (
        endpoint.state.ready
        if endpoint.state
        else None
    )

    # Current configuration-update state
    config_update_state = (
        endpoint.state.config_update
        if endpoint.state
        else None
    )

    print(
        "Ready:",
        ready_state,
        "| Config update:",
        config_update_state,
    )

    # Deployment is fully complete only when:
    # 1. Endpoint is READY
    # 2. Configuration is no longer updating
    if (
        ready_state is not None
        and config_update_state is not None
        and str(ready_state).endswith("READY")
        and str(config_update_state).endswith(
            "NOT_UPDATING"
        )
    ):
        print(
            "Endpoint deployment is complete."
        )
        break

    time.sleep(
        poll_interval_seconds
    )

    elapsed += (
        poll_interval_seconds
    )

else:
    raise TimeoutError(
        "Serving endpoint did not become "
        "ready within the expected time."
    )

##### 9. Refresh and inspect

In [0]:
endpoint = (
    workspace_client
    .serving_endpoints
    .get(
        SERVING_ENDPOINT_NAME
    )
)

print(
    "Endpoint:",
    endpoint.name
)

print(
    "Ready state:",
    endpoint.state.ready
)

for entity in (
    endpoint.config.served_entities
    or []
):

    print(
        "Entity:",
        entity.name
    )

    print(
        "Model:",
        entity.entity_name
    )

    print(
        "Version:",
        entity.entity_version
    )

    print(
    "Config update:",
    endpoint.state.config_update
)

In [0]:
endpoint = (
    workspace_client.serving_endpoints.get(SERVING_ENDPOINT_NAME)
)

print(
    "Endpoint:",
    endpoint.name
)

print(
    "Ready state:",
    endpoint.state.ready
)

print(
    "Config update:",
    endpoint.state.config_update
)

In [0]:
for entity in (
    endpoint.config.served_entities
    or []
):

    print(
        "Entity name:",
        entity.name
    )

    print(
        "Model:",
        entity.entity_name
    )

    print(
        "Version:",
        entity.entity_version
    )

    print(
        "Workload size:",
        entity.workload_size
    )

    print(
        "Scale to zero:",
        entity.scale_to_zero_enabled
    )

    print("-" * 60)

``` text

Endpoint: support-ticket-transformer-endpoint

Ready: READY

Entity: support-ticket-transformer

Model:dbw_agentic_ai_dev.support_ticket_ai.support_ticket_transformer_classifier

Version:2

Workload size:Small

Scale to zero:True

```

##### 10. Deployment verification

In [0]:
assert (
    endpoint.name
    == SERVING_ENDPOINT_NAME
)

assert endpoint.config is not None

assert (
    endpoint.config.served_entities
)

served_entity = (
    endpoint.config
    .served_entities[0]
)

assert (
    served_entity.entity_name
    == REGISTERED_MODEL_NAME
)

assert (
    str(
        served_entity.entity_version
    )
    == candidate_version
)

print(
    "Model deployment verification passed."
)

Notice what we are not testing yet:

 POST request
- → raw ticket
- → endpoint
- → prediction

That belongs in Notebook 15 — endpoint_testing.

Keeping that separation makes the notebooks cleaner.

##### Key Learnings

Registration and deployment are separate lifecycle stages.

- Registered Model → governed model artifact
- Serving Endpoint→ running infrastructure that exposes that model

The endpoint serves a specific model version. We are not deploying the abstract registered-model name alone. The endpoint ultimately resolves to a concrete version.

The Candidate alias gives us indirection.

Notebook 14 asks: Which version is Candidate? rather than: Deploy Version 1 forever.

So later:

Candidate → Version 2 allows the deployment workflow to discover Version 2 dynamically.

served_entities is the current API concept. A serving endpoint can host one or more served entities, and traffic can be routed among them.

Scale-to-zero is useful for learning environments. It reduces idle serving consumption, although the first request after idle can experience startup latency.

##### Conclusion


Notebook 14 deployed the Unity Catalog Candidate version of the support-ticket Transformer to a Databricks Model Serving endpoint. The model version was resolved dynamically through its registry alias rather than hard-coded, preserving a cleaner deployment lifecycle.

The notebook was designed to be rerunnable: it creates the endpoint when absent and updates the existing endpoint when present. Deployment readiness and served-entity configuration were verified before completing the workflow.

The model is now registered, governed and hosted behind Databricks-managed serving infrastructure. The next step is to validate the external inference contract by sending raw support-ticket requests to the endpoint.